In [8]:
import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
import pandas as pd
import os
import warnings
from dataclasses import dataclass
from tqdm import tqdm

warnings.filterwarnings('ignore')

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"

MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]

POSITIVE_CONTROLS = [
    ("760.5856", "761.5864"), ("788.6156", "789.6178"), 
    ("810.5997", "811.6054"), ("792.632", "793.5559"),
    ("813.6858", "814.6872"), ("524.3706", "525.3731"),
    ("522.3557", "523.3569"), ("734.5686", "735.572"),
    ("947.6513", "948.6564"), ("560.3341", "561.3381"),
    ("971.6514", "972.6575"), ("504.3442", "506.3606"), 
    ("484.2952", "485.3011"), ("627.534", "628.5358"), 
    ("455.2904", "456.2954"), ("455.2904", "457.3071"),
    ("666.4319", "667.4356"), ("678.4684", "679.4734"), 
    ("703.574", "704.5774"), ("706.5388", "707.5393")
]

NEGATIVE_CONTROLS = [
    ("760.5856", "810.5997"), ("788.6156", "798.5951"), 
    ("810.5997", "792.632"), ("792.632", "524.3706"),
    ("813.6858", "522.3557"), ("524.3706", "947.6513"),
    ("740.4725", "739.5955"), ("739.4661", "740.6021"),
    ("629.5422", "630.6165"), ("579.5304", "592.3947"), 
    ("759.6355", "760.5856"), ("772.6211", "773.5294"),
    ("788.6156", "797.5895"), ("840.641", "842.6652"), 
    ("848.5591", "848.6185"), ("848.5591", "848.6907"),
    ("868.6629", "870.6936"), ("971.6514", "971.6854") 
]

FOLD_CHANGE = 10

@dataclass
class SpatialSignature:
    mz: str
    values: np.ndarray
    histogram: np.ndarray
    importance: np.ndarray
    morans: float  # <--- Defined as 'morans'

# =============================================================================
# 2. METRIC EXTRACTION
# =============================================================================

def get_8_metrics(s1: SpatialSignature, s2: SpatialSignature) -> dict:
    # 1. Intensity Ratio Consistency
    mask = (s1.values > np.percentile(s1.values, 10)) & (s2.values > np.percentile(s2.values, 10))
    r_con = 0
    if mask.sum() > 10:
        rats = np.minimum(s1.values[mask], s2.values[mask]) / (np.maximum(s1.values[mask], s2.values[mask]) + 1e-8)
        r_con = 1 / (1 + (np.std(rats) / (np.mean(rats) + 1e-8)))
    
    # Primary Metrics
    v_corr = max(0, pearsonr(s1.values, s2.values)[0])
    p1, p2 = s1.values >= np.percentile(s1.values, 80), s2.values >= np.percentile(s2.values, 80)
    peak = (p1 & p2).sum() / ((p1 | p2).sum() + 1e-8)
    
    # Importance Metrics
    imp_iou = ((s1.importance > 0.5) & (s2.importance > 0.5)).sum() / ((s1.importance > 0.5) | (s2.importance > 0.5)).sum()
    i_corr = max(0, pearsonr(s1.importance, s2.importance)[0])
    
    # Descriptor Metrics
    h_corr = max(0, pearsonr(s1.histogram.flatten(), s2.histogram.flatten())[0])
    val_iou = np.minimum(s1.values, s2.values).sum() / (np.maximum(s1.values, s2.values).sum() + 1e-8)
    m_sim = 1 / (1 + abs(s1.morans - s2.morans)) # <--- FIXED: matches data class
    
    return {
        'intensity_ratio_consistency': r_con, 
        'value_correlation': v_corr, 
        'peak_colocalization': peak, 
        'importance_iou': imp_iou, 
        'importance_correlation': i_corr, 
        'spatial_hist_corr': h_corr, 
        'value_iou': val_iou, 
        'morans_similarity': m_sim
    }

# =============================================================================
# 3. CROSS-VALIDATED ML CALIBRATOR
# =============================================================================

class CrossValidatedCalibrator:
    def get_sig(self, adata, mz, nn_idx):
        v = adata[:, mz].X.toarray().flatten() if hasattr(adata.X, 'toarray') else adata[:, mz].X.flatten()
        coords = np.column_stack([adata.obs['x_um'], adata.obs['y_um']])
        hist = np.histogram2d(coords[:,0], coords[:,1], bins=10, weights=v)[0]
        imp = np.var(v[nn_idx[:, 1:]], axis=1)
        # Placeholder for Moran's I calculation - typically 0.5 similarity if not calculated
        return SpatialSignature(mz, v, hist, imp, 0.5)

    def run(self):
        print("--- PHASE 1: COLLECTING SPATIAL DATA ACROSS ALL COHORTS ---")
        data_rows = []
        for file_name in tqdm(MSI_SAMPLE_FILES):
            path = os.path.join(MSI_INPUT_FOLDER, file_name)
            adata = sc.read_h5ad(path)
            coords = np.column_stack([adata.obs['x_um'], adata.obs['y_um']])
            nn_idx = NearestNeighbors(n_neighbors=9).fit(coords).kneighbors(coords)[1]

            for m1, m2 in POSITIVE_CONTROLS + NEGATIVE_CONTROLS:
                if m1 in adata.var_names and m2 in adata.var_names:
                    s1, s2 = self.get_sig(adata, m1, nn_idx), self.get_sig(adata, m2, nn_idx)
                    metrics = get_8_metrics(s1, s2)
                    metrics['label'] = 1 if (m1, m2) in POSITIVE_CONTROLS else 0
                    metrics['sample_id'] = file_name
                    data_rows.append(metrics)

        df = pd.DataFrame(data_rows)
        samples = df['sample_id'].unique()
        
        ordered_features = [
            'intensity_ratio_consistency', 'value_correlation', 'peak_colocalization',
            'importance_iou', 'importance_correlation', 'spatial_hist_corr',
            'value_iou', 'morans_similarity'
        ]
        
        X_full = df[ordered_features]
        y_full = df['label']

        print(f"\n--- PHASE 2: {FOLD_CHANGE}-FOLD CROSS-VALIDATION (BY ANIMAL) ---")
        kf = KFold(n_splits=FOLD_CHANGE, shuffle=True, random_state=42)
        fold_aucs = []
        fold_weights = []

        for train_idx, test_idx in kf.split(samples):
            train_samples = samples[train_idx]
            test_samples = samples[test_idx]
            
            X_train = df[df['sample_id'].isin(train_samples)][ordered_features]
            y_train = df[df['sample_id'].isin(train_samples)]['label']
            X_test = df[df['sample_id'].isin(test_samples)][ordered_features]
            y_test = df[df['sample_id'].isin(test_samples)]['label']

            model = LogisticRegression(fit_intercept=False, penalty='l2')
            model.fit(X_train, y_train)

            coeffs = np.abs(model.coef_[0])
            normalized_w = coeffs / np.sum(coeffs)
            fold_weights.append(normalized_w)

            y_probs = model.predict_proba(X_test)[:, 1]
            fold_aucs.append(roc_auc_score(y_test, y_probs))

        avg_weights = np.mean(fold_weights, axis=0)
        std_weights = np.std(fold_weights, axis=0)
        avg_auc = np.mean(fold_aucs)

        self.print_thesis_report(ordered_features, avg_weights, std_weights, avg_auc)

    def print_thesis_report(self, names, avg_w, std_w, auc):
        import numpy as np
        
        # Calculate Standard Error (SE = SD / sqrt(n_folds))
        se_w = std_w / np.sqrt(FOLD_CHANGE)
        
        print("\n" + "="*95)
        print("PHD THESIS CALIBRATION REPORT: ROBUST SPATIAL WEIGHTS")
        print("="*95)
        print(f"{FOLD_CHANGE}-Fold Cross-Validated AUC: {auc:.4f}")
        print("\nValidated Weights (Fractional Mean [0-1] +/- SD and SE):")
        print("-" * 95)
        print(f"{'Metric Name':<35} | {'Weight (0-1)':<14} | {'SD':<10} | {'Std. Error (SE)':<15}")
        print("-" * 95)
        for i in range(len(names)):
            print(f"{names[i]:<35} | {avg_w[i]:>12.4f}   | {std_w[i]:>8.4f}   | {se_w[i]:>12.4f}")
        print("-" * 95)
        print(f"{'TOTAL (Sum)':<35} | {np.sum(avg_w):>12.4f}   |")
        print("="*95)
        print(f"Justification: Standard Error (SE) represents the precision of the weight estimate across")
        print(f"all {FOLD_CHANGE} folds. Low SE indicates high confidence in the global weight distribution.")
        print("="*95)

if __name__ == "__main__":
    calibrator = CrossValidatedCalibrator()
    calibrator.run()

--- PHASE 1: COLLECTING SPATIAL DATA ACROSS ALL COHORTS ---


100%|██████████| 16/16 [00:05<00:00,  2.90it/s]


--- PHASE 2: 10-FOLD CROSS-VALIDATION (BY ANIMAL) ---

PHD THESIS CALIBRATION REPORT: ROBUST SPATIAL WEIGHTS
10-Fold Cross-Validated AUC: 0.9621

Validated Weights (Fractional Mean [0-1] +/- SD and SE):
-----------------------------------------------------------------------------------------------
Metric Name                         | Weight (0-1)   | SD         | Std. Error (SE)
-----------------------------------------------------------------------------------------------
intensity_ratio_consistency         |       0.0376   |   0.0020   |       0.0006
value_correlation                   |       0.0325   |   0.0057   |       0.0018
peak_colocalization                 |       0.1025   |   0.0087   |       0.0028
importance_iou                      |       0.1490   |   0.0019   |       0.0006
importance_correlation              |       0.1649   |   0.0060   |       0.0019
spatial_hist_corr                   |       0.2072   |   0.0050   |       0.0016
value_iou                         